# Writing an Exercise

This notebook writes a multiple-choice exercise on the median of an even number of values, previews its versions, and checks that its choices never coincide.

In [ ]:
using Random, Statistics, Printf

fmt(x, digits) = Printf.format(Printf.Format("%.$(digits)f"), x)  # fixed decimals

## The Exercise as a Function

From a random number generator, the function draws eight distinct test scores and returns the question, the choices (the correct one first) and the explanation. The wrong choices are the usual mistakes: one of the two middle values, or the mean.

In [ ]:
function exercise(rng)
    x = shuffle(rng, sort(randperm(rng, 60)[1:8] .+ 10))
    s = sort(x)
    question = "Eight students scored $(join(x, ", ")) in a test. What is the median score?"
    choices = [fmt(median(x), 1), fmt(s[4], 1), fmt(s[5], 1), fmt(mean(x), 1)]
    explanation = "Sorted: $(join(s, ", ")). The median is the mean of the 4th and 5th values: ($(s[4]) + $(s[5])) / 2 = $(fmt(median(x), 1))."
    return (; question, choices, explanation)
end

## Previewing Versions

Each seed gives a version. The choices are shuffled with the same generator, after the data, so the correct one moves between the letters.

In [ ]:
for seed in 1:3
    rng = Xoshiro(seed)
    version = exercise(rng)
    order = randperm(rng, length(version.choices))
    println(version.question)
    for (letter, i) in zip("ABCD", order)
        println("  ", letter, ") ", version.choices[i], i == 1 ? "   (correct)" : "")
    end
    println()
end

## Choices That Coincide

The median can equal the mean, for instance when the scores are symmetric. Count the versions, among the first 1000, with two equal choices.

In [ ]:
ambiguous = [seed for seed in 1:1000 if !allunique(exercise(Xoshiro(seed)).choices)]
length(ambiguous)

If some versions are ambiguous, skip them: draw the version again with another seed until the choices differ.

In [ ]:
function clear_version(seed)
    while !allunique(exercise(Xoshiro(seed)).choices)
        seed += 1000
    end
    return exercise(Xoshiro(seed))
end

all(allunique(clear_version(seed).choices) for seed in 1:1000)